In [1]:
import pandas as pd
from category_encoders import TargetEncoder 

# loadin the data
train_df = pd.read_csv(r"Y:\code\regression-pipeline\data\processed\cleaned_train.csv")
val_df = pd.read_csv(r"Y:\code\regression-pipeline\data\processed\cleaned_val.csv")
holdout_df = pd.read_csv(r"Y:\code\regression-pipeline\data\raw\holdout.csv")

# settings for pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = '{:.2f}'.format

print(f"train is from  {train_df["date"].min()} and {train_df["date"].max()}")
print(f"val is from  {val_df["date"].min()} and {val_df["date"].max()}")
print(f"holdout is from  {holdout_df["date"].min()} and {holdout_df["date"].max()}")

# converting to datetime object
train_df["date"] = pd.to_datetime(train_df["date"])
val_df["date"] = pd.to_datetime(val_df["date"])
holdout_df["date"] = pd.to_datetime(holdout_df["date"])

train is from  2012-03-31 and 2019-12-31
val is from  2020-01-31 and 2021-12-31
holdout is from  2022-01-31 and 2023-12-31


In [2]:
# date features

def add_date_features(df: pd.DataFrame) -> pd.DataFrame:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["quarter"] = df["date"].dt.quarter
    
    df.insert(1, "year", df.pop("year"))
    df.insert(2, "month", df.pop("month"))
    df.insert(3, "quarter", df.pop("quarter"))
    return df

train_df = add_date_features(train_df)
val_df = add_date_features(val_df)
holdout_df = add_date_features(holdout_df)

In [3]:
# zipcode frequencies
zip_counts = train_df["zipcode"].value_counts()

train_df["zipcode_freqs"] = train_df["zipcode"].map(zip_counts).fillna(0)
val_df["zipcode_freqs"] = val_df["zipcode"].map(zip_counts).fillna(0)
holdout_df["zipcode_freqs"] = holdout_df["zipcode"].map(zip_counts).fillna(0)

In [4]:
# target encoding
te = TargetEncoder(cols=["city_full"])

train_df["city_encoded"] = te.fit_transform(train_df["city_full"], train_df["price"])
val_df["city_encoded"] = te.transform(val_df["city_full"])
holdout_df["city_encoded"] = te.transform(holdout_df["city_full"])

In [8]:
# drop unsued columns
# we drop median_sale_price to prevent data leakage
drop_cols = ["date", "city_full", "city", "zipcode", "median_sale_price"]

train_df.drop(columns=drop_cols, inplace=True, errors="ignore")
val_df.drop(columns=drop_cols, inplace=True, errors="ignore")
holdout_df.drop(columns=drop_cols, inplace=True, errors="ignore")

In [9]:
# saving the dfs'

train_df.to_csv(r"Y:\code\regression-pipeline\data\processed\fe_train.csv" , index=False)
val_df.to_csv(r"Y:\code\regression-pipeline\data\processed\fe_val.csv" , index=False)
holdout_df.to_csv(r"Y:\code\regression-pipeline\data\processed\fe_holdout.csv" , index=False)

print("Feature engineering complete")

Feature engineering complete
